# NexaMem Quick Start with Azure Cache for Redis

> **📋 Note**: This comprehensive notebook has been split into focused versions for easier use:
> 
> - **🚀 [Basic Authentication](quickstart_azure_redis_basic.ipynb)** - Access keys (recommended for getting started)
> - **🏢 [Enterprise Authentication](quickstart_azure_redis_enterprise.ipynb)** - Entra ID, Service Principal, Key Vault (recommended for production)
>
> **Choose the notebook that best fits your use case!**

---

This notebook demonstrates how to use **NexaMem** (Next-Generation AI Memory) with **Azure Cache for Redis** for scalable, persistent chatbot memory management.

## What is NexaMem?

NexaMem is a Python package for managing AI chatbot conversation history with:
- Multiple storage backends (memory, file, SQLite, Redis)
- Channel-based memory organization
- Advanced features like encryption, quotas, and TTL
- Azure Cloud integration

## Prerequisites

Before running this notebook, ensure you have:
1. An Azure Cache for Redis instance
2. Redis connection details (hostname, access key, or Entra ID credentials)
3. Required Python packages installed

## 1. Install Required Dependencies

We'll install only the external dependencies and use the **local development version** of NexaMem from this repository. This ensures you have the latest features and bug fixes that may not yet be published to PyPI.

**What we'll install:**
- `redis` - Python Redis client  
- `azure-identity` & `azure-keyvault-secrets` - Optional Azure authentication packages

**What we'll use locally:**
- `nexamem` - From the `src/nexamem/` directory in this repository

In [8]:
# Install dependencies and use local development version of NexaMem
# Only install Redis dependency from pip, use local NexaMem from repo
!pip install redis

# Optional: Azure identity packages for Entra ID authentication
!pip install azure-identity azure-keyvault-secrets

print("✅ Dependencies installed!")
print("ℹ️  Using local development version of NexaMem from repository")
print("   This ensures you have the latest features and fixes")

## 2. Import Required Modules

In [ ]:
import os
import sys
import json
from typing import Dict, Any

# Add local src directory to path for development version
repo_root = os.getcwd()
src_path = os.path.join(repo_root, 'src')

print(f"📁 Repository root: {repo_root}")
print(f"📁 Source path: {src_path}")
print(f"📁 Source exists: {os.path.exists(src_path)}")

if os.path.exists(src_path):
    # Insert at beginning of path to prioritize local version
    if src_path not in sys.path:
        sys.path.insert(0, src_path)
        print(f"✅ Added {src_path} to Python path")
    
    # Force reload of modules if they're already loaded to ensure we get the latest local version
    modules_to_reload = ['nexamem', 'nexamem.redis_adapter', 'nexamem.history', 'nexamem.aimemory', 'nexamem.config']
    for module in modules_to_reload:
        if module in sys.modules:
            del sys.modules[module]
            print(f"🔄 Reloaded module: {module}")
else:
    print("⚠️  Local src directory not found - will try to use installed package")

# NexaMem imports from local development version
try:
    from nexamem import initialize, ChatHistory, AIMemory, MemoryScope
    from nexamem.redis_adapter import AzureRedisConnectionHelper
    
    print("✅ All imports successful!")
    
    # Show which version we're actually using
    import nexamem
    if hasattr(nexamem, '__file__'):
        nexamem_path = nexamem.__file__
        print(f"📍 Using NexaMem from: {nexamem_path}")
        
        # Check if it's the local development version
        if src_path.lower() in nexamem_path.lower():
            print("🚀 Confirmed: Using LOCAL DEVELOPMENT version")
        else:
            print("⚠️  Warning: Using installed package version")
    
    # Show Redis adapter source
    import nexamem.redis_adapter
    if hasattr(nexamem.redis_adapter, '__file__'):
        print(f"📍 Redis adapter from: {nexamem.redis_adapter.__file__}")
        
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("   Make sure the src/nexamem directory exists and contains the modules")

✅ All imports successful!
📍 Using NexaMem from: c:\Users\chadada\source\repos\nexamem\src\nexamem\__init__.py
📍 Redis adapter from: c:\Users\chadada\source\repos\nexamem\src\nexamem\redis_adapter.py


## 3. Configuration Setup

### Option A: Using Access Keys (Recommended for Getting Started)

Replace the placeholders with your actual Azure Redis details:

In [6]:
# Azure Redis Configuration with Access Key
# TODO: Replace these placeholders with your actual Azure Redis details

AZURE_REDIS_CONFIG = {
    "history_storage": "redis",
    "redis_config": {
        # Replace with your Azure Redis hostname
        "azure_hostname": "YOUR-REDIS-NAME.redis.cache.windows.net",
        
        # Replace with your primary or secondary access key from Azure Portal
        "azure_access_key": "YOUR-REDIS-ACCESS-KEY-HERE",
        
        # Optional: Database number (default: 0)
        "db": 0,
        
        # Optional: Custom key prefix for organization
        "key_prefix": "nexamem:chatbot",
        
        # Optional: Connection timeout settings
        "socket_timeout": 5.0,
        "socket_connect_timeout": 5.0
    },
    "debug": True
}

print("📋 Configuration prepared (remember to update placeholders!)")
print(f"   Hostname: {AZURE_REDIS_CONFIG['redis_config']['azure_hostname']}")
print(f"   Key prefix: {AZURE_REDIS_CONFIG['redis_config']['key_prefix']}")

📋 Configuration prepared (remember to update placeholders!)
   Hostname: YOUR-REDIS-NAME.redis.cache.windows.net
   Key prefix: nexamem:chatbot


### Option B: Using Environment Variables (Production Recommended)

In [ ]:
# Alternative: Environment-based configuration for production
# Set these environment variables in your deployment:
# - AZURE_REDIS_HOSTNAME
# - AZURE_REDIS_ACCESS_KEY

def create_config_from_env():
    """Create Redis configuration from environment variables."""
    hostname = os.getenv('AZURE_REDIS_HOSTNAME')
    access_key = os.getenv('AZURE_REDIS_ACCESS_KEY')
    
    if not hostname or not access_key:
        print("❌ Missing environment variables:")
        print("   - AZURE_REDIS_HOSTNAME")
        print("   - AZURE_REDIS_ACCESS_KEY")
        return None
    
    return {
        "history_storage": "redis",
        "redis_config": {
            "azure_hostname": hostname,
            "azure_access_key": access_key,
            "db": 0,
            "key_prefix": "nexamem:production"
        },
        "debug": False
    }

ENV_CONFIG = create_config_from_env()
if ENV_CONFIG:
    print("✅ Environment-based configuration ready")
else:
    print("ℹ️  Environment variables not set - using manual configuration")

### Option C: Entra ID Authentication (Enterprise)

In [ ]:
# Enterprise: Entra ID (Azure AD) Authentication
ENTRA_ID_CONFIG = {
    "history_storage": "redis",
    "redis_config": {
        "azure_hostname": "YOUR-REDIS-NAME.redis.cache.windows.net",
        "use_azure_entra_id": True,
        
        # For managed identity (recommended)
        "azure_username": "your-client-id-or-username",
        
        # For service principal (uncomment if needed)
        # "azure_client_id": "your-client-id",
        # "azure_client_secret": "your-client-secret",
        # "azure_tenant_id": "your-tenant-id",
        
        "key_prefix": "nexamem:enterprise"
    },
    "debug": True
}

print("🏢 Enterprise Entra ID configuration prepared")

## 4. Testing Connection with Helper Utilities

Before initializing NexaMem, let's test the Redis connection configuration:

In [7]:
# Test Azure Redis connection configuration
try:
    # Get Redis configuration using the helper
    redis_config = AzureRedisConnectionHelper.get_azure_redis_config(
        hostname="YOUR-REDIS-NAME.redis.cache.windows.net",  # Replace with your hostname
        access_key="YOUR-ACCESS-KEY-HERE"  # Replace with your access key
    )
    
    print("✅ Redis configuration generated successfully:")
    for key, value in redis_config.items():
        if key == 'password':
            print(f"   {key}: {'*' * 8}...{value[-4:] if len(value) > 4 else '***'}")
        else:
            print(f"   {key}: {value}")
            
except Exception as e:
    print(f"❌ Configuration error: {e}")
    print("   Make sure to replace placeholders with actual values")

✅ Redis configuration generated successfully:
   host: YOUR-REDIS-NAME.redis.cache.windows.net
   port: 6380
   ssl: True
   ssl_cert_reqs: None
   decode_responses: True
   password: ********...HERE


## 5. Initialize NexaMem with Azure Redis

Now let's initialize NexaMem with your Azure Redis configuration:

In [ ]:
# Choose which configuration to use
# Use ENV_CONFIG if environment variables are set, otherwise use manual config
config_to_use = ENV_CONFIG if ENV_CONFIG else AZURE_REDIS_CONFIG

try:
    # Initialize NexaMem with Azure Redis
    initialize(config_to_use)
    print("🎉 NexaMem successfully initialized with Azure Redis!")
    
    # Test the connection
    history = ChatHistory()
    print("✅ ChatHistory instance created successfully")
    
except Exception as e:
    print(f"❌ Initialization failed: {e}")
    print("\n🔧 Troubleshooting tips:")
    print("   1. Verify your Azure Redis hostname and access key")
    print("   2. Check network connectivity to Azure")
    print("   3. Ensure Redis allows SSL connections (port 6380)")
    print("   4. Verify firewall rules allow your IP address")

## 6. Basic ChatHistory Usage

Let's demonstrate basic chatbot conversation management:

In [ ]:
# Set up session context
history = ChatHistory()

# Configure the conversation context
history.set_session_context(
    agent_id="azure_assistant",
    user_id="user_demo_123",
    session_id="quickstart_session",
    env="demo"
)

print("👤 Session context configured:")
print(f"   Agent: azure_assistant")
print(f"   User: user_demo_123")
print(f"   Session: quickstart_session")
print(f"   Environment: demo")

In [ ]:
# Add conversation messages
print("💬 Adding conversation messages...\n")

# User messages
history.add_user_message(
    "Hello! I'm testing NexaMem with Azure Redis.", 
    tokens=10
)

# Assistant responses
history.add_assistant_message(
    "Great! I can see you're successfully connected to Azure Cache for Redis. "
    "Your conversation history is now being stored in Azure's managed Redis service.",
    tokens=25
)

history.add_user_message(
    "What are the benefits of using Azure Redis for chatbot memory?",
    tokens=12
)

history.add_assistant_message(
    "Azure Cache for Redis provides several benefits:\n"
    "1. **Scalability**: Handle thousands of concurrent conversations\n"
    "2. **Persistence**: Data survives application restarts\n"
    "3. **Performance**: Sub-millisecond latency for memory operations\n"
    "4. **Security**: Built-in encryption and Azure security features\n"
    "5. **Reliability**: 99.9% SLA with automatic failover",
    tokens=60
)

print("✅ Messages added to conversation history")

In [ ]:
# Retrieve and display conversation history
messages = history.get_history()

print(f"📚 Retrieved {len(messages)} messages from Azure Redis:\n")

for i, msg in enumerate(messages, 1):
    role_icon = "👤" if msg['role'] == 'user' else "🤖"
    print(f"{i}. {role_icon} **{msg['role'].title()}** ({msg.get('tokens', 0)} tokens)")
    print(f"   {msg['content']}\n")

## 7. Advanced AIMemory Usage

Let's explore the advanced AIMemory features with channel-based organization:

In [ ]:
# Create channels configuration for advanced memory management
import tempfile

def create_channels_config():
    """Create a temporary channels configuration file."""
    channels_content = """
channels:
  working:
    ttl_sec: 14400      # 4 hours - short-term working memory
    encrypt: true
    quota_bytes: 50000  # 50KB limit

  longterm:
    ttl_sec: 604800     # 7 days - important information
    encrypt: true
    quota_bytes: 100000 # 100KB limit

  routing:
    ttl_sec: 86400      # 24 hours - conversation routing hints
    encrypt: false
    quota_bytes: 10000  # 10KB limit

  debug:
    ttl_sec: 3600       # 1 hour - debugging information
    encrypt: false
    quota_bytes: 5000   # 5KB limit
"""
    
    with tempfile.NamedTemporaryFile(mode='w', suffix='.yaml', delete=False) as f:
        f.write(channels_content.strip())
        return f.name

channels_file = create_channels_config()
print(f"📋 Created channels configuration: {channels_file}")

In [ ]:
# Initialize AIMemory with Azure Redis and channels
try:
    memory = AIMemory(
        channels_yaml=channels_file,
        # Azure Redis configuration
        azure_hostname="YOUR-REDIS-NAME.redis.cache.windows.net",  # Replace
        azure_access_key="YOUR-ACCESS-KEY-HERE"  # Replace
    )
    
    print("🧠 AIMemory initialized with Azure Redis!")
    
    # List available channels
    channels = memory.list_channels()
    print("\n📺 Available memory channels:")
    for name, config in channels.items():
        ttl_hours = config.ttl_sec // 3600
        quota_kb = config.quota_bytes // 1000 if config.quota_bytes else "unlimited"
        print(f"   {name}: {ttl_hours}h TTL, {quota_kb}KB quota, encrypt={config.encrypt}")
    
except Exception as e:
    print(f"❌ AIMemory initialization failed: {e}")
    print("   Make sure to replace hostname and access key placeholders")

In [ ]:
# Create memory scope for organized data access
scope = MemoryScope(
    agent_id="financial_advisor",
    user_id="client_456",
    session_id="portfolio_review_2024",
    env="production"
)

print(f"🎯 Memory scope created:")
print(f"   Agent: {scope.agent_id}")
print(f"   User: {scope.user_id}")
print(f"   Session: {scope.session_id}")
print(f"   Environment: {scope.env}")

In [ ]:
# Write different types of information to different channels
print("✍️  Writing to different memory channels...\n")

try:
    # Working memory - current conversation context
    working_id = memory.write(
        scope=scope,
        channel="working",
        content="Client is interested in diversifying their investment portfolio. "
                "Current allocation: 70% stocks, 20% bonds, 10% cash. "
                "Risk tolerance: moderate. Investment horizon: 10 years."
    )
    print(f"💼 Working memory: {working_id[:8]}...")
    
    # Long-term memory - important client information
    longterm_id = memory.write(
        scope=scope,
        channel="longterm",
        content="Client profile: Age 45, married, 2 children. "
                "Annual income: $120K. Investment goal: retirement at 65. "
                "Prefers ESG investments. Has 401k with employer matching."
    )
    print(f"🏦 Long-term memory: {longterm_id[:8]}...")
    
    # Routing information - for conversation flow
    routing_id = memory.write(
        scope=scope,
        channel="routing",
        content=json.dumps({
            "topic": "portfolio_diversification",
            "confidence": 0.95,
            "next_agent": "portfolio_specialist",
            "urgency": "normal"
        })
    )
    print(f"🗺️  Routing info: {routing_id[:8]}...")
    
    # Debug information
    debug_id = memory.write(
        scope=scope,
        channel="debug",
        content="Session started at 2024-07-11 10:30 UTC. "
                "User agent: Chrome/91.0. IP: 192.168.1.100"
    )
    print(f"🐛 Debug info: {debug_id[:8]}...")
    
    print("\n✅ All memory writes successful!")
    
except Exception as e:
    print(f"❌ Memory write error: {e}")

In [ ]:
# Read back information from different channels
print("📖 Reading from memory channels...\n")

try:
    # Read working memory
    working_messages, working_meta = memory.read(
        scope=scope,
        channel="working",
        max_msgs=5
    )
    print(f"💼 Working memory ({working_meta['total_messages']} messages):")
    for msg in working_messages:
        print(f"   {msg[:100]}...")
    
    # Read long-term memory
    longterm_messages, longterm_meta = memory.read(
        scope=scope,
        channel="longterm",
        max_msgs=5
    )
    print(f"\n🏦 Long-term memory ({longterm_meta['total_messages']} messages):")
    for msg in longterm_messages:
        print(f"   {msg[:100]}...")
    
    # Read routing information
    routing_messages, routing_meta = memory.read(
        scope=scope,
        channel="routing",
        max_msgs=5
    )
    print(f"\n🗺️  Routing info ({routing_meta['total_messages']} messages):")
    for msg in routing_messages:
        try:
            parsed = json.loads(msg)
            print(f"   Topic: {parsed.get('topic')}, Next: {parsed.get('next_agent')}")
        except:
            print(f"   {msg[:100]}...")
    
except Exception as e:
    print(f"❌ Memory read error: {e}")

## 8. Memory Analytics and Monitoring

Let's explore memory usage and analytics:

In [ ]:
# Check memory usage across channels
print("📊 Memory Usage Analytics\n")

try:
    channels = memory.list_channels()
    
    total_bytes = 0
    for channel_name in channels.keys():
        try:
            byte_count = memory.count_bytes(scope, channel_name)
            total_bytes += byte_count
            
            # Get channel messages for count
            messages, meta = memory.read(scope, channel_name, max_msgs=1000)
            msg_count = meta['total_messages']
            
            print(f"📺 {channel_name:10} | {msg_count:3} msgs | {byte_count:6} bytes | {byte_count/1024:.1f} KB")
            
        except Exception as e:
            print(f"📺 {channel_name:10} | Error: {e}")
    
    print(f"\n📈 Total memory usage: {total_bytes} bytes ({total_bytes/1024:.1f} KB)")
    
except Exception as e:
    print(f"❌ Analytics error: {e}")

## 9. Connection Testing and Health Check

In [ ]:
# Test Redis connection health
try:
    # Access the underlying Redis adapter for health checks
    from nexamem.config import get_config
    
    config = get_config()
    if hasattr(config, 'redis_adapter') and config.redis_adapter:
        adapter = config.redis_adapter
        
        # Ping test
        ping_result = adapter.ping()
        print(f"🏓 Redis ping: {'✅ Success' if ping_result else '❌ Failed'}")
        
        # Connection info
        info = adapter.get_info()
        print(f"\n🔗 Connection Information:")
        print(f"   Host: {info.get('host', 'unknown')}")
        print(f"   Port: {info.get('port', 'unknown')}")
        print(f"   SSL: {info.get('ssl', 'unknown')}")
        print(f"   Database: {info.get('db', 'unknown')}")
        print(f"   Azure Redis: {info.get('is_azure', 'unknown')}")
        print(f"   Connected: {info.get('connected', 'unknown')}")
    else:
        print("❌ No Redis adapter found in configuration")
        
except Exception as e:
    print(f"❌ Health check error: {e}")

## 10. Cleanup and Best Practices

Finally, let's clean up our demo data and review best practices:

In [ ]:
# Clean up demo data (optional)
try:
    # Count data before cleanup
    total_before = 0
    for channel_name in memory.list_channels().keys():
        try:
            byte_count = memory.count_bytes(scope, channel_name)
            total_before += byte_count
        except:
            pass
    
    print(f"🧹 Data before cleanup: {total_before} bytes")
    
    # Bulk erase all data for this scope
    deleted_count = memory.bulk_erase(scope)
    print(f"🗑️  Deleted {deleted_count} keys from Redis")
    
    # Verify cleanup
    total_after = 0
    for channel_name in memory.list_channels().keys():
        try:
            byte_count = memory.count_bytes(scope, channel_name)
            total_after += byte_count
        except:
            pass
    
    print(f"📊 Data after cleanup: {total_after} bytes")
    print("✅ Demo cleanup completed!")
    
except Exception as e:
    print(f"❌ Cleanup error: {e}")

# Clean up temporary files
try:
    os.unlink(channels_file)
    print("🗑️  Temporary channels file removed")
except:
    pass

## 🎯 Best Practices Summary

### Security
- **Use Entra ID authentication** in production instead of access keys
- **Store secrets in Azure Key Vault** for enterprise deployments
- **Enable encryption** for sensitive channels
- **Use SSL connections** (automatic with Azure Redis)

### Performance
- **Set appropriate TTL values** to manage memory usage
- **Use quotas** to prevent runaway memory consumption
- **Configure connection pooling** for high-traffic applications
- **Monitor memory usage** regularly

### Organization
- **Use meaningful scope identifiers** (agent_id, user_id, session_id)
- **Organize data into logical channels** (working, longterm, routing, debug)
- **Set environment-specific configurations** (dev, staging, prod)
- **Use consistent key prefixes** for data organization

### Monitoring
- **Implement health checks** in your application
- **Monitor Redis metrics** in Azure Portal
- **Set up alerts** for connection failures or memory limits
- **Log memory operations** for debugging

## Next Steps

1. **Replace placeholder values** with your actual Azure Redis details
2. **Run the notebook** to test your configuration
3. **Integrate NexaMem** into your chatbot application
4. **Set up monitoring** and alerting for production use
5. **Consider Azure Key Vault** for credential management

## Resources

- [NexaMem Documentation](https://github.com/your-repo/nexamem)
- [Azure Cache for Redis Documentation](https://docs.microsoft.com/en-us/azure/azure-cache-for-redis/)
- [Redis Python Client Documentation](https://redis-py.readthedocs.io/)
- [Azure Identity Documentation](https://docs.microsoft.com/en-us/python/api/overview/azure/identity-readme)

---

*Happy coding with NexaMem and Azure Redis! 🚀*